In [1]:
### Cell 1 – Installs & Imports
!pip install --quiet tqdm

import os, glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# GPU setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on", device)
torch.backends.cudnn.benchmark = True


Running on cuda


In [2]:
### Cell 2 – Hyperparameters & Paths
DATA_DIR    = "/content/shape2"
SAMPLES_DIR = "/content/generated_samples"
os.makedirs(SAMPLES_DIR, exist_ok=True)

batch_size    = 16
image_size    = 200     # no resizing
latent_dim    = 64      # more capacity
num_epochs    = 40
learning_rate = 1e-3
beta          = 4.0     # KL weight
alpha_l1      = 0.5     # L1 vs BCE tradeoff


In [3]:
### Cell 3 – Dataset & DataLoader
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),  # [0,255]→[0,1]
])

class ShapeDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.files = [
            p for p in glob.glob(f"{folder}/*")
            if p.lower().endswith((".png",".jpg",".jpeg"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("L")
        if self.transform:
            img = self.transform(img)
        return img, 0  # dummy label

dataset = ShapeDataset(DATA_DIR, transform=transform)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                     num_workers=2, pin_memory=True)
print("Loaded", len(dataset), "images")


Loaded 900 images


In [4]:
### Cell 4 – Improved β-VAE (4 conv layers + correct dims + Sigmoid decoder)
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        # Encoder: 200→100→50→25→12
        self.enc = nn.Sequential(
            nn.Conv2d(1,  32, 4, 2, 1), nn.ReLU(True),  # →100
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(True),  # →50
            nn.Conv2d(64,128, 4, 2, 1), nn.ReLU(True),  # →25
            nn.Conv2d(128,256,4, 2, 1), nn.ReLU(True),  # →12
            nn.Flatten()
        )
        flat_dim = 256 * 12 * 12
        self.fc_mu     = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)

        # Decoder: 12→25→50→100→200
        self.fc_dec = nn.Linear(latent_dim, flat_dim)
        self.dec    = nn.Sequential(
            # 12→25 (needs output_padding=1)
            nn.ConvTranspose2d(256,128,4,2,1,output_padding=1), nn.ReLU(True),
            nn.ConvTranspose2d(128,64, 4,2,1),                  nn.ReLU(True),  #25→50
            nn.ConvTranspose2d(64, 32, 4,2,1),                  nn.ReLU(True),  #50→100
            nn.ConvTranspose2d(32,  1, 4,2,1),                  nn.Sigmoid()    #100→200
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 256, 12, 12)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

model     = VAE(latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
print(model)


VAE(
  (enc): Sequential(
    (0): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Flatten(start_dim=1, end_dim=-1)
  )
  (fc_mu): Linear(in_features=36864, out_features=64, bias=True)
  (fc_logvar): Linear(in_features=36864, out_features=64, bias=True)
  (fc_dec): Linear(in_features=64, out_features=36864, bias=True)
  (dec): Sequential(
    (0): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): ConvTranspose2d(64, 32, kernel_siz

In [5]:
### Cell 5 – Combined BCE + L₁ + β·KL Loss
import torch.nn.functional as F

def vae_loss(recon_x, x, mu, logvar):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    l1  = F.l1_loss(recon_x, x, reduction='sum')
    kl  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + alpha_l1 * l1 + beta * kl


In [6]:
### Cell 6 – Training Loop
model.train()
for epoch in range(1, num_epochs+1):
    total_loss = 0
    for imgs, _ in loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(imgs)
        loss = vae_loss(recon, imgs, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs} — Avg Loss: {total_loss/len(dataset):.4f}")

    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            batch, _ = next(iter(loader))
            batch = batch.to(device)
            rec, _, _ = model(batch)
            comp = torch.cat([batch[:8], rec[:8]])
            save_image(comp, f"{SAMPLES_DIR}/recon_ep{epoch}.png", nrow=8)
            z    = torch.randn(16, latent_dim, device=device)
            samp = model.decode(z)
            save_image(samp, f"{SAMPLES_DIR}/sample_ep{epoch}.png", nrow=4)
        model.train()


Epoch 1/40 — Avg Loss: 15064.0133
Epoch 2/40 — Avg Loss: 3668.0410
Epoch 3/40 — Avg Loss: 3565.8370
Epoch 4/40 — Avg Loss: 3554.8361
Epoch 5/40 — Avg Loss: 3546.0582
Epoch 6/40 — Avg Loss: 3530.4312
Epoch 7/40 — Avg Loss: 3512.9608
Epoch 8/40 — Avg Loss: 3486.5282
Epoch 9/40 — Avg Loss: 3331.0362
Epoch 10/40 — Avg Loss: 2906.2112
Epoch 11/40 — Avg Loss: 2611.3884
Epoch 12/40 — Avg Loss: 2434.9734
Epoch 13/40 — Avg Loss: 2289.0446
Epoch 14/40 — Avg Loss: 2175.3511
Epoch 15/40 — Avg Loss: 2111.0223
Epoch 16/40 — Avg Loss: 2043.5895
Epoch 17/40 — Avg Loss: 1951.8803
Epoch 18/40 — Avg Loss: 1915.6297
Epoch 19/40 — Avg Loss: 1900.8025
Epoch 20/40 — Avg Loss: 1805.4570
Epoch 21/40 — Avg Loss: 1774.7447
Epoch 22/40 — Avg Loss: 1731.5324
Epoch 23/40 — Avg Loss: 1728.2187
Epoch 24/40 — Avg Loss: 1674.1871
Epoch 25/40 — Avg Loss: 1640.7217
Epoch 26/40 — Avg Loss: 1623.7229
Epoch 27/40 — Avg Loss: 1591.3733
Epoch 28/40 — Avg Loss: 1589.7777
Epoch 29/40 — Avg Loss: 1545.5808
Epoch 30/40 — Avg Loss

In [11]:
### Cell 7 – Encode Reference Shape
from PIL import Image
import os

# Path to your reference
ref_path = os.path.join(DATA_DIR, "original2.png")

# Load, transform, encode
ref_img     = Image.open(ref_path).convert("L")
ref_tensor  = transform(ref_img).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    mu_ref, _ = model.encode(ref_tensor)      # (1, latent_dim)
mu_ref = mu_ref.cpu().numpy()[0]              # (latent_dim,)
print("Reference µ vector shape:", mu_ref.shape)


Reference µ vector shape: (64,)


In [12]:
### Cell 8 – Compute Latent‐Space Distances
import numpy as np

scores = []  # list of (path, distance)
model.eval()
with torch.no_grad():
    for img_path in dataset.files:
        img   = Image.open(img_path).convert("L")
        tensor= transform(img).unsqueeze(0).to(device)
        mu, _ = model.encode(tensor)
        mu    = mu.cpu().numpy()[0]
        dist  = float(np.linalg.norm(mu - mu_ref))
        scores.append((img_path, dist))

# sort ascending: lowest distance = most similar
scores.sort(key=lambda p: p[1])
print("Top 5 closest:", scores[:5])


Top 5 closest: [('/content/shape2/original2.png', 0.0), ('/content/shape2/10386.png', 3.675779104232788), ('/content/shape2/10548.png', 4.711287021636963), ('/content/shape2/10659.png', 4.868171215057373), ('/content/shape2/10731.png', 5.1106953620910645)]


In [15]:
### Cell Z – Generate Self-Contained HTML Ranking

import os, base64

html = [
    "<!DOCTYPE html>",
    "<html><head><meta charset='utf-8'><title>Similarity Ranking</title></head><body>",
    "<h1>Shape Similarity Ranking (to original2.png)</h1>",
    "<ol style='list-style:none;'>"
]

for path, dist in scores:
    ext = os.path.splitext(path)[1].lower().lstrip(".")
    mime = f"image/{'png' if ext=='png' else 'jpeg'}"
    with open(path, "rb") as imgf:
        b64 = base64.b64encode(imgf.read()).decode("utf-8")
    html.append(
        f"<li style='margin-bottom:20px;'>"
        f"<img src='data:{mime};base64,{b64}' width='200'><br>"
        f"<strong>Distance:</strong> {dist:.4f}"
        f"</li>"
    )

html += ["</ol></body></html>"]

out_file = "/content/similarity_ranking.html"
with open(out_file, "w") as f:
    f.write("\n".join(html))

print("Wrote self-contained HTML to", out_file)


Wrote self-contained HTML to /content/similarity_ranking.html
